# UZH-FPV + Spyx New Architectures

This notebook targets the UZH-FPV Drone Racing dataset (https://fpv.ifi.uzh.ch/).

Status in Tonic:
- No direct `tonic.datasets` class for UZH-FPV in this environment.

Goal:
- Use local UZH-FPV files when present.
- Run architecture ablations with new Spyx neuron/model families.

In [ ]:
from pathlib import Path

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

import spyx.models as fm

DATA_ROOT = Path("../data/UZH_FPV")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT.resolve())


def tonic_has_uzh_fpv():
    import tonic.datasets as d
    names = {n.lower() for n in dir(d)}
    return any(k in names for k in ["uzhfpv", "fpv", "uzh_fpv"])

print("tonic_has_uzh_fpv =", tonic_has_uzh_fpv())
print("expected dirs from uzh_fpv_open:", ["calib", "raw", "output"])

In [ ]:
def synthetic_event_batch(batch=4, T=16, H=64, W=64, C=2, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.poisson(0.02, size=(T, batch, H, W, C)).astype(np.float32)
    return jnp.asarray(x)


def make_architectures(input_hw=(64, 64), input_channels=2, output_dim=6):
    conv_cfg = fm.ConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=24, channels2=48, output_dim=output_dim)
    sparse_cfg = fm.SparseConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=24, channels2=48, output_dim=output_dim, event_threshold=0.0)
    dw_cfg = fm.DepthwiseSepConvConfig(input_hw=input_hw, input_channels=input_channels, depth_multiplier1=1, pointwise1=24, depth_multiplier2=1, pointwise2=48, output_dim=output_dim)

    return {
        "conv_lif": lambda x: fm.ConvLIFSNN(conv_cfg)(x),
        "ternary_conv_lif": lambda x: fm.TernaryConvLIFSNN(conv_cfg)(x),
        "sparse_event_conv_lif": lambda x: fm.SparseEventConvLIFSNN(sparse_cfg)(x),
        "depthwise_sep_conv_lif": lambda x: fm.DepthwiseSeparableConvLIFSNN(dw_cfg)(x),
    }


x = synthetic_event_batch()
models = make_architectures(output_dim=6)

for name, fn in models.items():
    transformed = hk.without_apply_rng(hk.transform(fn))
    params = transformed.init(jax.random.PRNGKey(0), x)
    y, aux = transformed.apply(params, x)
    print(name, "->", y.shape, "spike_rate:", np.asarray(aux["spike_rate"]))

## Next Steps for Real UZH-FPV Runs

1. Place real dataset content in `../data/UZH_FPV` with `calib`, `raw`, and `output` folders.
2. Add an event parser for mDAVIS streams (e.g. converted `.npz` tensor cache).
3. Replace synthetic batch generation with cached event frames and run full train/eval splits.